# Classificação de Superfícies de Vias — Voxar Labs PS 2025

**Autor:** Alisson da Silva Bernadino  
**Data:** Abril de 2026  

---

## Identificação da Abordagem

Este notebook resolve o desafio de classificação de imagens em 3 classes (**Asphalt**, **Belgian Blocks**, **Off-road**) utilizando **Transfer Learning** com PyTorch.

**Modelo escolhido:** ResNet-18 pré-treinado no ImageNet  
**Justificativa:** Dado o tempo sugerido de 6–10h, treinar um modelo do zero seria inviável. A ResNet-18 é uma arquitetura pequena (~11M parâmetros), amplamente validada para tarefas de classificação visual. O fine-tuning das camadas superiores adapta os filtros genéricos do ImageNet para as texturas específicas de superfícies de estrada — exatamente o que precisamos.

**Bibliotecas principais:**
- `torch` / `torchvision` — backbone, training loop, transforms
- `matplotlib` / `seaborn` — visualizações e matriz de confusão
- `scikit-learn` — métricas (F1, classification report)
- `numpy` / `PIL` — manipulação de arrays e imagens

**Estrutura do notebook:**
1. Entendimento do Problema (EDA)
2. Solução Baseline
3. Experimentos
4. Resultados Consolidados
5. Análise Crítica
6. Uso de Ferramentas

---
## Seção 1 — Entendimento do Problema (EDA & Estratégia)

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import defaultdict

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Caminho base do dataset — ajuste se necessário
BASE_DIR = Path('../dataset_processed')
SPLITS   = ['train', 'test']
CLASSES  = ['asphalt', 'belgian_blocks', 'offroad']
CLASS_LABELS = {'asphalt': 'Asphalt', 'belgian_blocks': 'Belgian Blocks', 'offroad': 'Off-road'}

print('Dataset path:', BASE_DIR.resolve())
print('Exists:', BASE_DIR.exists())

### 1.1 Natureza do Problema

Trata-se de um problema de **classificação de imagens multiclasse** (3 classes) com características que elevam sua complexidade:

| Desafio | Descrição |
|---|---|
| **Desbalanceamento severo** | A classe *Asphalt* domina o dataset; *Belgian Blocks* é a menor |
| **Variação visual intra-classe** | Iluminação noturna, chuva, sujeira, diferentes câmeras |
| **Similaridade inter-classes** | Off-road e Belgian Blocks podem ter texturas próximas |
| **Ruído de captura** | Diferentes ângulos, resoluções e artefatos de compressão |

In [ ]:
# ── 1.2 Distribuição das Classes ──────────────────────────────────────────────
counts = {}
for split in SPLITS:
    counts[split] = {}
    for cls in CLASSES:
        path = BASE_DIR / split / cls
        counts[split][cls] = len(list(path.glob('*')))

print('=== Distribuição do Dataset ===')
total_train = sum(counts['train'].values())
total_test  = sum(counts['test'].values())

print(f"\n{'Classe':<20} {'Train':>8} {'%Train':>8} {'Test':>8} {'%Test':>8}")
print('-' * 55)
for cls in CLASSES:
    tr = counts['train'][cls]
    te = counts['test'][cls]
    print(f"{CLASS_LABELS[cls]:<20} {tr:>8} {tr/total_train*100:>7.1f}% {te:>8} {te/total_test*100:>7.1f}%")
print('-' * 55)
print(f"{'TOTAL':<20} {total_train:>8} {'100.0%':>8} {total_test:>8} {'100.0%':>8}")

# Razão de desbalanceamento
max_cls = max(counts['train'], key=counts['train'].get)
min_cls = min(counts['train'], key=counts['train'].get)
ratio = counts['train'][max_cls] / counts['train'][min_cls]
print(f"\nRazão de desbalanceamento (train): {ratio:.1f}x ({CLASS_LABELS[max_cls]} / {CLASS_LABELS[min_cls]})")

In [ ]:
# ── Gráfico de distribuição ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4A90D9', '#E67E22', '#27AE60']
labels = [CLASS_LABELS[c] for c in CLASSES]

for ax, split in zip(axes, SPLITS):
    vals = [counts[split][c] for c in CLASSES]
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'Distribuição — {split.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Número de imagens')
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                str(v), ha='center', va='bottom', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Desbalanceamento do Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

**Observação:** O dataset apresenta desbalanceamento severo (~7x entre Asphalt e Belgian Blocks no treino). Isso significa que um modelo ingênuo pode atingir **~73% de acurácia simplesmente prevendo sempre Asphalt** — motivo pelo qual usaremos **F1-Score macro** e **Matriz de Confusão por classe** como métricas primárias.

### 1.3 Definição das Métricas de Sucesso

| Métrica | Por quê usar |
|---|---|
| **Acurácia** | Referência geral, mas enganosa com classes desbalanceadas |
| **F1-Score macro** | Métrica primária: trata todas as classes com igual peso |
| **Precision / Recall por classe** | Identifica onde o modelo falha especificamente |
| **Matriz de Confusão** | Evidência visual dos padrões de erro |

In [ ]:
# ── 1.4 Visualização de Amostras por Classe ───────────────────────────────────
def load_random_samples(split, cls, n=4, seed=SEED):
    path = BASE_DIR / split / cls
    files = list(path.glob('*'))
    random.seed(seed)
    chosen = random.sample(files, min(n, len(files)))
    return [Image.open(f).convert('RGB') for f in chosen]

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
fig.suptitle('Amostras do Conjunto de Treino por Classe', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASSES):
    imgs = load_random_samples('train', cls, n=4)
    for col, img in enumerate(imgs):
        ax = axes[row][col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_LABELS[cls], fontsize=11, fontweight='bold',
                          rotation=0, labelpad=60, va='center')

plt.tight_layout()
plt.savefig('eda_class_samples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 1.5 Casos Difíceis — Análise Qualitativa ─────────────────────────────────
# Busca imagens escuras (brilho médio baixo) como proxy para "noite/chuva"
def get_image_stats(split, cls, n_sample=30):
    path = BASE_DIR / split / cls
    files = list(path.glob('*'))
    random.seed(SEED)
    sample = random.sample(files, min(n_sample, len(files)))
    stats = []
    for f in sample:
        img = np.array(Image.open(f).convert('RGB'))
        stats.append({'file': f, 'brightness': img.mean(), 'std': img.std()})
    return stats

print('=== Análise de Brilho por Classe (treino) ===')
fig, ax = plt.subplots(figsize=(10, 4))
all_brightness = []
all_labels = []

for cls in CLASSES:
    stats = get_image_stats('train', cls, n_sample=60)
    brightness = [s['brightness'] for s in stats]
    all_brightness.append(brightness)
    all_labels.append(CLASS_LABELS[cls])
    print(f"  {CLASS_LABELS[cls]:<18}: mean={np.mean(brightness):.1f}, min={np.min(brightness):.1f}, max={np.max(brightness):.1f}")

bp = ax.boxplot(all_brightness, labels=all_labels, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribuição de Brilho por Classe\n(valores baixos = imagens escuras/noite/chuva)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Brilho médio (0–255)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('eda_brightness_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Exibir exemplos de imagens "difíceis" (mais escuras) ──────────────────────
def get_dark_samples(split, cls, n=3, brightness_threshold=80):
    path = BASE_DIR / split / cls
    files = list(path.glob('*'))
    dark = []
    for f in files:
        img = np.array(Image.open(f).convert('RGB'))
        if img.mean() < brightness_threshold:
            dark.append((f, img.mean()))
    dark.sort(key=lambda x: x[1])
    return [Image.open(f).convert('RGB') for f, _ in dark[:n]]

fig, axes = plt.subplots(3, 3, figsize=(11, 9))
fig.suptitle('Casos Difíceis — Imagens de Baixo Brilho (noite/chuva)', fontsize=13, fontweight='bold')

for row, cls in enumerate(CLASSES):
    dark_imgs = get_dark_samples('train', cls, n=3)
    # preenche com imagens aleatórias se não houver imagens escuras suficientes
    if len(dark_imgs) < 3:
        extra = load_random_samples('train', cls, n=3-len(dark_imgs), seed=1)
        dark_imgs.extend(extra)
    for col, img in enumerate(dark_imgs[:3]):
        ax = axes[row][col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(CLASS_LABELS[cls], fontsize=11, fontweight='bold',
                          rotation=0, labelpad=60, va='center')

plt.tight_layout()
plt.savefig('eda_hard_cases.png', dpi=120, bbox_inches='tight')
plt.show()

### 1.6 Resumo dos Desafios Identificados

1. **Desbalanceamento crítico (~7x):** A classe *Belgian Blocks* tem apenas 94 amostras no treino. Um modelo treinado sem estratégia tende a ignorar essa classe.

2. **Variação intra-classe extrema:** A análise de brilho mostra que existem imagens muito escuras em todas as classes, especialmente *Off-road*, o que simula condições noturnas e de chuva.

3. **Ambiguidade inter-classes:** Belgian Blocks e Off-road compartilham características texturais irregulares, tornando-as difíceis de separar em baixa iluminação.

4. **Heterogeneidade de captura:** Diferentes dispositivos, ângulos e distâncias focais geram variância adicional não relacionada ao conteúdo semântico.

**Estratégia proposta:**
- Transfer Learning com ResNet-18 (pré-treinado ImageNet)
- Weighted CrossEntropy para tratar desbalanceamento
- Data Augmentation focado em variações de iluminação/cor
- Avaliação primária por F1-Score macro

In [ ]:
# ── 1.7 Análise de Dimensões e Formatos ───────────────────────────────────────
print('=== Amostra de dimensões de imagens (treino) ===')
dim_stats = defaultdict(list)

for cls in CLASSES:
    path = BASE_DIR / 'train' / cls
    files = list(path.glob('*'))
    random.seed(SEED)
    sample = random.sample(files, min(20, len(files)))
    for f in sample:
        try:
            img = Image.open(f)
            dim_stats[cls].append(img.size)  # (width, height)
        except Exception:
            pass

for cls in CLASSES:
    dims = dim_stats[cls]
    widths  = [d[0] for d in dims]
    heights = [d[1] for d in dims]
    print(f"  {CLASS_LABELS[cls]:<18}: W=[{min(widths)}–{max(widths)}]  H=[{min(heights)}–{max(heights)}]")

print('\nAs imagens têm dimensões variadas → resize para 224x224 necessário antes do modelo.')

---
## Seção 2 — Solução Baseline
*Esta seção será implementada na branch `feature/02-baseline-model`.*

---
## Seção 3 — Experimentos
*Esta seção será implementada nas branches `exp/01-class-weighting` e `exp/02-robust-augmentation`.*

---
## Seção 4 — Resultados Consolidados
*Esta seção será consolidada na branch `main` após todos os experimentos.*

---
## Seção 5 — Análise Crítica
*Esta seção será escrita na etapa final.*

---
## Seção 6 — Uso de Ferramentas

Este projeto contou com suporte do LLM **Google Gemini / Antigravity** (Claude Sonnet) para:
- Estruturação inicial do notebook e sugestão do fluxo de análise
- Revisão de código e sugestões de boas práticas em PyTorch
- Apoio na redação técnica das seções analíticas

Todo raciocínio, escolhas de abordagem, hipóteses dos experimentos e interpretação dos resultados são de responsabilidade da autora.